In [1]:
# Cell 1: Import Libraries (CORRECTED)
import ee
import geemap
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image
import pandas as pd
import folium

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [2]:
# Cell 2: Authenticate and Initialize Earth Engine (CORRECTED)
try:
    # Initialize Earth Engine
    ee.Initialize(project="ee-kgoutham386")  # Use your actual project ID
    print("✅ Earth Engine initialized successfully!")
except Exception as e:
    print("🔐 Authentication required...")
    ee.Authenticate()  # This will open browser for login
    ee.Initialize(project="ee-kgoutham386")
    print("✅ Earth Engine initialized after authentication!")

✅ Earth Engine initialized successfully!


In [3]:
# Cell 3: Define Bangalore Study Area (ENHANCED for Urban Morphology)
# Bangalore bounding box - larger area for urban analysis
bangalore_bbox = ee.Geometry.Rectangle([
    77.46, 12.87,  # Southwest (long, lat)
    77.72, 13.07   # Northeast (long, lat)
])

# Create interactive map centered on Bangalore
Map = geemap.Map(center=[12.97, 77.59], zoom=11)
Map.addLayer(bangalore_bbox, {'color': 'red'}, 'Study Area - Bangalore')
Map

Map(center=[12.97, 77.59], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [4]:
# Cell 4: Load Sentinel-2 Data (CORRECTED for Urban Analysis)
print("🛰️ Loading Sentinel-2 data for urban morphology analysis...")

# Use Sentinel-2 Surface Reflectance data
collection = ee.ImageCollection("COPERNICUS/S2_SR") \
    .filterBounds(bangalore_bbox) \
    .filterDate('2022-01-01', '2022-12-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))  # Low cloud cover

print(f"📊 Total Sentinel-2 images found: {collection.size().getInfo()}")

# Get median composite to reduce noise
median_composite = collection.median()

# Add to map with urban-optimized visualization
urban_vis_params = {
    'bands': ['B4', 'B3', 'B2'],  # True color: Red, Green, Blue
    'min': 0,
    'max': 3000
}

Map.addLayer(median_composite, urban_vis_params, 'Sentinel-2 Urban Base')
Map

🛰️ Loading Sentinel-2 data for urban morphology analysis...
📊 Total Sentinel-2 images found: 23


Map(center=[12.97, 77.59], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [5]:
# Cell 5: Inspect Image Metadata for Urban Analysis
print("🔍 Inspecting image metadata for urban features...")

first_image = collection.first()

# Get image properties
properties = first_image.getInfo()['properties']
print("📋 Key Metadata for Urban Analysis:")
print(f"   - Acquisition Date: {properties.get('DATE_ACQUIRED', 'N/A')}")
print(f"   - Cloud Cover: {properties.get('CLOUDY_PIXEL_PERCENTAGE', 'N/A')}%")
print(f"   - Sun Elevation: {properties.get('SUN_ELEVATION', 'N/A')}°")

# Band information for urban feature extraction
band_info = first_image.getInfo()['bands']
print(f"\n🎯 Available Bands ({len(band_info)} total):")

urban_bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']  # Key bands for urban analysis
for band in band_info:
    if band['id'] in urban_bands:
        print(f"   - {band['id']}: {band.get('data_type', {}).get('precision', 'N/A')}")

🔍 Inspecting image metadata for urban features...
📋 Key Metadata for Urban Analysis:
   - Acquisition Date: N/A
   - Cloud Cover: 1.28577%
   - Sun Elevation: N/A°

🎯 Available Bands (23 total):
   - B2: int
   - B3: int
   - B4: int
   - B8: int
   - B11: int
   - B12: int


In [6]:
# Cell 6: Cloud Masking Function (CORRECTED)
print("☁️ Applying cloud masking for clean urban analysis...")

def mask_clouds_sentinel2(image):
    """Mask clouds and shadows in Sentinel-2 imagery"""
    # Cloud mask from QA60 band (bits 10 and 11)
    qa = image.select('QA60')
    cloud_bitmask = 1 << 10
    cirrus_bitmask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bitmask).eq(0) \
            .And(qa.bitwiseAnd(cirrus_bitmask).eq(0))
    
    return image.updateMask(mask).divide(10000)  # Scale reflectance to 0-1

# Apply cloud masking
collection_clean = collection.map(mask_clouds_sentinel2)
median_clean = collection_clean.median()

# Add cloud-masked layer
Map.addLayer(median_clean, urban_vis_params, 'Cloud-Masked Urban')
Map

print("✅ Cloud masking applied successfully!")

☁️ Applying cloud masking for clean urban analysis...
✅ Cloud masking applied successfully!


In [7]:
# Cell 7: Calculate NDVI for Green Space Analysis (CORRECTED)
print("🌿 Calculating NDVI for urban green space analysis...")

def calculate_ndvi(image):
    """Calculate Normalized Difference Vegetation Index"""
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return image.addBands(ndvi)

# Apply NDVI calculation
collection_ndvi = collection_clean.map(calculate_ndvi)
ndvi_median = collection_ndvi.select('NDVI').median()

# Add NDVI layer with urban-focused colormap
ndvi_vis_params = {
    'min': -0.2,  # Urban areas typically have low/negative NDVI
    'max': 0.8,   # Healthy vegetation
    'palette': ['brown', 'yellow', 'green']  # Urban to vegetation
}

Map.addLayer(ndvi_median, ndvi_vis_params, 'NDVI - Urban Green Spaces')
Map

print("✅ NDVI calculated for vegetation analysis!")

🌿 Calculating NDVI for urban green space analysis...
✅ NDVI calculated for vegetation analysis!


In [ ]:
# Cell 8: Calculate NDBI for Built-up Area Analysis
print("🏢 Calculating NDBI for built-up area detection...")

def calculate_ndbi(image):
    """Calculate Normalized Difference Built-up Index"""
    # NDBI = (SWIR1 - NIR) / (SWIR1 + NIR)
    ndbi = image.normalizedDifference(['B11', 'B8']).rename('NDBI')
    return image.addBands(ndbi)

# Apply NDBI calculation
collection_ndbi = collection_clean.map(calculate_ndbi)
ndbi_median = collection_ndbi.select('NDBI').median()

# Add NDBI layer
ndbi_vis_params = {
    'min': -0.5,
    'max': 0.5,
    'palette': ['blue', 'cyan', 'red']  # Water to built-up areas
}

Map.addLayer(ndbi_median, ndbi_vis_params, 'NDBI - Built-up Areas')
Map

print("✅ NDBI calculated for urban built-up analysis!");

🏢 Calculating NDBI for built-up area detection...
✅ NDBI calculated for urban built-up analysis!


In [9]:
# Cell 9: Calculate Urban Index Composite
print("🏙️ Creating urban morphology composite...")

def calculate_urban_composite(image):
    """Create composite index for urban morphology"""
    ndvi = image.normalizedDifference(['B8', 'B4'])
    ndbi = image.normalizedDifference(['B11', 'B8'])
    
    # Urban Index (simplified)
    urban_index = ndbi.subtract(ndvi).rename('URBAN_INDEX')
    return image.addBands(urban_index)

collection_urban = collection_clean.map(calculate_urban_composite)
urban_median = collection_urban.select('URBAN_INDEX').median()

urban_vis = {
    'min': -1,
    'max': 1,
    'palette': ['green', 'yellow', 'red', 'darkred']  # Vegetation to dense urban
}

Map.addLayer(urban_median, urban_vis, 'Urban Morphology Index')
Map

print("✅ Urban morphology composite created!")

🏙️ Creating urban morphology composite...
✅ Urban morphology composite created!


In [10]:
# Cell 10: Export Data for ML Training
print("💾 Exporting data for machine learning pipeline...")

# Export the cloud-masked median composite
export_task_rgb = ee.batch.Export.image.toDrive(
    image=median_clean,
    description='bangalore_2022_urban_rgb',
    folder='GENURBAN_DATA',
    fileNamePrefix='bangalore_2022_rgb',
    scale=10,  # 10m resolution
    region=bangalore_bbox,
    maxPixels=1e9
)

# Export NDVI for vegetation analysis
export_task_ndvi = ee.batch.Export.image.toDrive(
    image=ndvi_median,
    description='bangalore_2022_ndvi',
    folder='GENURBAN_DATA', 
    fileNamePrefix='bangalore_2022_ndvi',
    scale=10,
    region=bangalore_bbox,
    maxPixels=1e9
)

# Export Urban Index for morphology analysis
export_task_urban = ee.batch.Export.image.toDrive(
    image=urban_median,
    description='bangalore_2022_urban_index',
    folder='GENURBAN_DATA',
    fileNamePrefix='bangalore_2022_urban_index', 
    scale=10,
    region=bangalore_bbox,
    maxPixels=1e9
)

# Start exports (uncomment when ready)
# export_task_rgb.start()
# export_task_ndvi.start() 
# export_task_urban.start()

print("📤 Export tasks configured. Uncomment to start downloading to Google Drive.")
print("📍 Files will be saved in: GENURBAN_DATA folder in your Google Drive")

💾 Exporting data for machine learning pipeline...
📤 Export tasks configured. Uncomment to start downloading to Google Drive.
📍 Files will be saved in: GENURBAN_DATA folder in your Google Drive


In [11]:
# Cell 11: Urban Feature Statistics
print("📊 Calculating urban feature statistics...")

# Calculate statistics for the study area
def calculate_statistics(image, geometry):
    stats = image.reduceRegion(
        reducer=ee.Reducer.mean().combine(
            reducer2=ee.Reducer.stdDev(), 
            sharedInputs=True
        ),
        geometry=geometry,
        scale=100,  # 100m scale for urban analysis
        bestEffort=True
    )
    return stats

# Get statistics for key bands
urban_stats = calculate_statistics(urban_median, bangalore_bbox)
ndvi_stats = calculate_statistics(ndvi_median, bangalore_bbox)
ndbi_stats = calculate_statistics(ndbi_median, bangalore_bbox)

print("🏙️ URBAN FEATURE STATISTICS:")
print(f"   Urban Index - Mean: {urban_stats.getInfo().get('URBAN_INDEX_mean', 'N/A'):.3f}")
print(f"   Urban Index - Std: {urban_stats.getInfo().get('URBAN_INDEX_stdDev', 'N/A'):.3f}")
print(f"   NDVI - Mean: {ndvi_stats.getInfo().get('NDVI_mean', 'N/A'):.3f}") 
print(f"   NDBI - Mean: {ndbi_stats.getInfo().get('NDBI_mean', 'N/A'):.3f}")

print("\n✅ Urban morphology baseline established!")

📊 Calculating urban feature statistics...
🏙️ URBAN FEATURE STATISTICS:
   Urban Index - Mean: -0.173
   Urban Index - Std: 0.149
   NDVI - Mean: 0.190
   NDBI - Mean: 0.016

✅ Urban morphology baseline established!
